# 09 · 激活与损失：softmax 与交叉熵

> **本节属于 Part 4 · 神经网络抽象 nn。**

要训练分类器，还差两块拼图：把网络输出变成**概率**的 `softmax`，以及衡量"预测概率与真实标签差距"的**交叉熵损失**。本节重点讲两件容易踩坑的事——**数值稳定性**，以及交叉熵那个异常漂亮的梯度 `softmax - onehot`。

## 学习目标

- 实现**数值稳定**的 `softmax`（减最大值技巧）与 `log_softmax`
- 实现融合了 softmax 的**交叉熵** `cross_entropy`，并理解其梯度 `(softmax - onehot)/N`
- 用数值梯度 + PyTorch 双重验证
- 了解 `MSELoss` 与激活 Module

## 1. softmax 与数值稳定性

softmax 把一排实数变成概率： $\text{softmax}(x)_i = e^{x_i} / \sum_j e^{x_j}$。

**陷阱**：如果 $x$ 很大，$e^{x}$ 会**溢出**成 `inf`。解决办法：所有元素先减去最大值再 exp——结果不变（分子分母同乘一个常数），但绝不溢出。

In [ ]:
import inspect
import numpy as np
from minitorch import Tensor
from minitorch.functional import softmax, log_softmax, cross_entropy
from minitorch.utils import numerical_gradient, rel_error

big = np.array([1000.0, 1001.0, 1002.0])
# 朴素做法：直接 exp -> 溢出
e = np.exp(big)
print("朴素 softmax:", e / e.sum(), " <- 溢出成 nan/inf")
# minitorch 的稳定实现
print("稳定 softmax:", softmax(Tensor(big)).data)

In [ ]:
print(inspect.getsource(softmax))

## 2. 交叉熵损失

交叉熵 = $-\frac{1}{N}\sum_i \log p_{i, y_i}$，其中 $p$ 是 softmax 概率，$y_i$ 是第 $i$ 个样本的正确类别。

我们用 `log_softmax`（同样用减最大值保证稳定）算出对数概率，再挑出正确类别那一项。看真实实现：

In [ ]:
print(inspect.getsource(cross_entropy))

### 那个漂亮的梯度：softmax − onehot

交叉熵对 logits 的梯度有极简形式：

$$\frac{\partial L}{\partial \text{logits}} = \frac{1}{N}(\text{softmax}(\text{logits}) - \text{onehot}(y))$$

直觉：把"预测概率"推向"真实分布"。我们的 autograd 会自动算出它——验证一下是否真等于这个公式：

In [ ]:
np.random.seed(0)
N, C = 8, 4
logits_np = np.random.randn(N, C)
targets = np.random.randint(0, C, size=N)

logits = Tensor(logits_np)
cross_entropy(logits, targets).backward()

# 手算 (softmax - onehot)/N
p = softmax(Tensor(logits_np)).data
onehot = np.zeros_like(p); onehot[np.arange(N), targets] = 1
manual = (p - onehot) / N
print("autograd 梯度 vs (softmax-onehot)/N 相对误差:", rel_error(logits.grad, manual))

# 再与数值梯度核对
def loss_np(z):
    z = z - z.max(axis=-1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(axis=-1, keepdims=True))
    return -logp[np.arange(N), targets].mean()
print("autograd 梯度 vs 数值梯度 相对误差:", rel_error(logits.grad, numerical_gradient(loss_np, logits_np.copy())))

## 3. PyTorch 对照

PyTorch 的 `F.cross_entropy` 同样**把 softmax 融进损失**（输入是原始 logits，不要自己先 softmax）。损失值与梯度都应一致：

In [ ]:
import torch
lt = torch.tensor(logits_np, requires_grad=True)
loss_t = torch.nn.functional.cross_entropy(lt, torch.tensor(targets))
loss_t.backward()

loss_ours = cross_entropy(Tensor(logits_np), targets)
print(f"损失:  ours={float(loss_ours.data):.6f}  torch={loss_t.item():.6f}")
print("梯度相对误差 (ours vs torch):", rel_error(manual, lt.grad.numpy()))

## 4. MSELoss 与激活 Module

回归用 `MSELoss`；激活也包装成了 Module（`ReLU/Sigmoid/Tanh/Softmax`），方便放进 `Sequential`。

In [ ]:
from minitorch import nn
mse = nn.MSELoss()
pred = Tensor(np.array([[1.0, 2.0], [3.0, 4.0]]))
tgt = Tensor(np.array([[1.5, 2.0], [2.0, 5.0]]))
print("MSE:", float(mse(pred, tgt).data))

relu = nn.ReLU()
print("ReLU([-1,0,2]):", relu(Tensor([-1.0, 0.0, 2.0])).data)

## 📦 沉淀进 minitorch

`softmax / log_softmax / cross_entropy` 在 **`minitorch/functional.py`**；`MSELoss / CrossEntropyLoss` 与激活 Module 在 **`minitorch/nn/`**。全部通过 `tests/test_nn.py`（含与 PyTorch 对照）。

## 小练习

1. **手写 softmax 反向**：不借助 autograd，手推 softmax 的雅可比，验证 `gradcheck`（提示：$\partial p_i/\partial x_j = p_i(\delta_{ij} - p_j)$）。
2. **为什么融合**：分别用"先 softmax 再取 log"和"直接 log_softmax"对一批很大的 logits 求交叉熵，比较数值稳定性。
3. **标签平滑（进阶）**：把 onehot 目标改成 `0.9` 正确类 + `0.1/C` 其他类，观察损失与梯度的变化。

## 小结 & 下一站

✅ 我们实现了数值稳定的 softmax/交叉熵，验证了 `softmax - onehot` 这一漂亮梯度，分类的最后一块拼图就位。

**下一站 → `10_init_and_mlp_on_mnist`**：讲清**权重初始化**为什么重要，然后用 minitorch **真刀真枪训练一个 MLP 识别 MNIST 手写数字**（目标 95%+），并与 PyTorch 对照。